# Code-golf RL with Modal sandboxes

This tutorial mirrors the PR's "code-golf with sandboxed tests" idea, but uses
`modal-training-gym` directly.

Workflow:
1. Create a tiny code-golf dataset.
2. Score model outputs by executing generated code inside Modal sandboxes.
3. Use that scorer both for offline eval and as SLIME `custom_rm_function`.
4. Train and compare base vs. trained behavior.

The key pattern is: **correctness first, size second**.
A sample only gets a size bonus if sandbox tests pass.

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@main

In [ ]:
import json
import re
from functools import lru_cache

import modal

from modal_training_gym import (
    DeploymentConfig,
    EvalConfig,
    EvalRowResult,
    HarborDataset,
    ModelDeployment,
    Qwen3_4B,
    SlimeRecipe,
    TrainConfig,
    WandbConfig,
)
from modal_training_gym.common.checkpoint import list_checkpoints

## Build Harbor tasks from MBPP

We ingest MBPP from Hugging Face and materialize a Harbor-style task tree:
- `instruction.md` for prompt text
- per-task metadata (`record.json`) for tests/reference solution

`HarborDataset` then converts those task folders into SLIME-ready chat rows.

In [ ]:
MBPP_REPO = "Muennighoff/mbpp"
MBPP_SPLIT = "train"
MBPP_LIMIT = 96
MBPP_TRAIN_SIZE = 80

def _extract_function_name(code: str) -> str | None:
    import ast

    try:
        tree = ast.parse(code)
    except SyntaxError:
        return None
    for node in tree.body:
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            return node.name
    return None

def _load_mbpp_records(limit: int) -> list[dict]:
    from datasets import load_dataset

    ds = load_dataset(MBPP_REPO, split=MBPP_SPLIT)
    ds = ds.select(range(min(limit, len(ds))))
    records = []
    for row in ds:
        task_id = int(row["task_id"])
        code = str(row["code"]).replace("\r\n", "\n").replace("\r", "\n")
        entrypoint = _extract_function_name(code)
        if not entrypoint:
            continue
        tests = [
            *list(row.get("test_list", [])),
            *list(row.get("challenge_test_list", [])),
        ]
        if not tests:
            continue
        records.append(
            {
                "task_id": task_id,
                "name": f"mbpp_{task_id:04d}",
                "entrypoint": entrypoint,
                "prompt": str(row["text"]).strip(),
                "reference_solution": code,
                "tests": tests,
            }
        )
    records.sort(key=lambda x: x["task_id"])
    if not records:
        raise ValueError("No usable MBPP records were loaded")
    return records

def _write_harbor_tasks(tasks_root: str, tasks: list[dict]) -> None:
    import os

    os.makedirs(tasks_root, exist_ok=True)
    for task in tasks:
        task_dir = os.path.join(tasks_root, task["name"])
        os.makedirs(task_dir, exist_ok=True)
        with open(os.path.join(task_dir, "instruction.md"), "w", encoding="utf-8") as f:
            f.write(
                (
                    "Solve the following Python task.\n\n"
                    f"Task:\n{task['prompt']}\n\n"
                    f"Required function name: `{task['entrypoint']}`.\n\n"
                    "Output only Python code. Do not include Markdown fences.\n"
                )
            )
        with open(os.path.join(task_dir, "record.json"), "w", encoding="utf-8") as f:
            json.dump(
                {
                    "task_id": task["task_id"],
                    "task_name": task["name"],
                    "entrypoint": task["entrypoint"],
                    "tests": task["tests"],
                    "reference_solution": task["reference_solution"],
                },
                f,
            )

class CodeGolfDataset(HarborDataset):
    apply_chat_template = True
    prompt_template = "{instruction}"
    label_metadata_path = "record.json"
    train_repeats = 16
    mbpp_limit = MBPP_LIMIT
    train_size = MBPP_TRAIN_SIZE

    def load(self):
        return _load_mbpp_records(limit=int(self.mbpp_limit))

    def prepare(self, path: str, eval_paths: dict[str, str] | None = None):
        import os

        tasks = self.load()
        tasks_root = os.path.join(os.path.dirname(path), "code_golf_tasks", "tasks")
        _write_harbor_tasks(tasks_root, tasks)
        self.task_root = tasks_root
        max_train = max(1, len(tasks) - 1)
        self.train_size = min(int(self.train_size), max_train)
        self.eval_repeats = 1
        super().prepare(path, eval_paths)


train_dataset = CodeGolfDataset()
eval_dataset = CodeGolfDataset()

## Sandbox-backed scorer

We execute candidate code in a Modal sandbox and run assert tests there.
Reward is:
- `0` if tests fail,
- otherwise `pass_rate * (1 + length_bonus_weight * size_bonus)`.

`size_bonus` is derived from `reference_bytes / candidate_bytes`, capped at `2.0`.

In [ ]:
_CODE_FENCE_RE = re.compile(r"```(?:python)?\s*(.*?)```", re.IGNORECASE | re.DOTALL)
_SANDBOX_IMAGE = modal.Image.debian_slim(python_version="3.11")
_LENGTH_BONUS_WEIGHT = 0.2

def extract_python_code(text: str) -> str:
    normalized = text.replace("\r\n", "\n").replace("\r", "\n").strip()
    if "<|im_start|>assistant" in normalized:
        normalized = normalized.rsplit("<|im_start|>assistant", 1)[-1]
    if "</think>" in normalized:
        normalized = normalized.split("</think>", 1)[-1]
    normalized = normalized.replace("<think>", "").replace("<|im_end|>", "").strip()
    if match := _CODE_FENCE_RE.search(normalized):
        return match.group(1).strip()
    return normalized

@lru_cache(maxsize=1)
def _sandbox_app() -> modal.App:
    return modal.App.lookup("training-gym-code-golf-rm", create_if_missing=True)

def _parse_label(sample) -> dict:
    raw = getattr(sample, "label", None)
    if not raw:
        return {}
    if isinstance(raw, dict):
        return raw
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {}

def _last_json_line(text: str) -> dict:
    for line in reversed(text.splitlines()):
        line = line.strip()
        if not line:
            continue
        try:
            data = json.loads(line)
        except json.JSONDecodeError:
            continue
        if isinstance(data, dict):
            return data
    return {}

def _build_sandbox_test_script(
    candidate_code: str,
    *,
    tests: list[str],
    entrypoint: str,
) -> str:
    lines = [
        "import json",
        f"candidate = {json.dumps(candidate_code)}",
        f"tests = {json.dumps(tests)}",
        f"entrypoint = {json.dumps(entrypoint)}",
        "scope = {}",
        "try:",
        "    exec(candidate, scope, scope)",
        "    fn = scope.get(entrypoint)",
        "    if not callable(fn):",
        "        raise RuntimeError(f'Missing callable: {entrypoint}')",
        "    passed = 0",
        "    for expr in tests:",
        "        exec(expr, scope, scope)",
        "        passed += 1",
        "    print(json.dumps({'pass_rate': passed / len(tests) if tests else 0.0, 'passed': passed, 'total': len(tests)}))",
        "except Exception as exc:",
        "    print(json.dumps({'pass_rate': 0.0, 'passed': 0, 'total': len(tests), 'error': repr(exc)}))",
    ]
    return "\n".join(lines) + "\n"

def score_code_with_sandbox(
    response: str,
    *,
    tests: list[str],
    entrypoint: str,
    reference_solution: str,
    timeout_sec: int = 25,
) -> tuple[float, dict]:
    candidate_code = extract_python_code(response)
    script = _build_sandbox_test_script(
        candidate_code,
        tests=tests,
        entrypoint=entrypoint,
    )

    try:
        sandbox = modal.Sandbox.create(
            "python",
            "-c",
            script,
            app=_sandbox_app(),
            image=_SANDBOX_IMAGE,
            timeout=timeout_sec,
            cpu=1.0,
            memory=1024,
        )
        stdout = sandbox.stdout.read()
        stderr = sandbox.stderr.read()
        exit_code = sandbox.wait()
    except Exception as exc:
        candidate_bytes = len(candidate_code.encode("utf-8"))
        reference_bytes = len(reference_solution.encode("utf-8"))
        return 0.0, {
            "pass_rate": 0.0,
            "candidate_bytes": candidate_bytes,
            "reference_bytes": reference_bytes,
            "ratio": 0.0,
            "sandbox_exit_code": -1,
            "sandbox_stderr": repr(exc),
        }

    payload = _last_json_line(stdout)
    pass_rate = float(payload.get("pass_rate", 0.0))

    candidate_bytes = len(candidate_code.encode("utf-8"))
    reference_bytes = len(reference_solution.encode("utf-8"))
    ratio = min(2.0, reference_bytes / max(candidate_bytes, 1))
    size_bonus = max(0.0, ratio - 1.0)
    reward = pass_rate * (1.0 + (_LENGTH_BONUS_WEIGHT * size_bonus))

    metadata = {
        "pass_rate": pass_rate,
        "candidate_bytes": candidate_bytes,
        "reference_bytes": reference_bytes,
        "ratio": ratio,
        "sandbox_exit_code": exit_code,
        "sandbox_stderr": stderr.strip(),
    }
    return reward, metadata

async def code_golf_rm(args, sample, **kwargs) -> float:
    import asyncio
    import json
    import re
    from functools import lru_cache

    import modal

    def _inline_score_code_with_sandbox(
        response: str,
        *,
        tests: list[str],
        entrypoint: str,
        reference_solution: str,
        timeout_sec: int = 25,
    ) -> tuple[float, dict]:
        code_fence_re = re.compile(r"```(?:python)?\s*(.*?)```", re.IGNORECASE | re.DOTALL)
        sandbox_image = modal.Image.debian_slim(python_version="3.11")
        length_bonus_weight = 0.2

        def extract_python_code(text: str) -> str:
            normalized = text.replace("\r\n", "\n").replace("\r", "\n").strip()
            if "<|im_start|>assistant" in normalized:
                normalized = normalized.rsplit("<|im_start|>assistant", 1)[-1]
            if "</think>" in normalized:
                normalized = normalized.split("</think>", 1)[-1]
            normalized = normalized.replace("<think>", "").replace("<|im_end|>", "").strip()
            if match := code_fence_re.search(normalized):
                return match.group(1).strip()
            return normalized

        @lru_cache(maxsize=1)
        def sandbox_app() -> modal.App:
            return modal.App.lookup("training-gym-code-golf-rm", create_if_missing=True)

        def last_json_line(text: str) -> dict:
            for line in reversed(text.splitlines()):
                line = line.strip()
                if not line:
                    continue
                try:
                    data = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if isinstance(data, dict):
                    return data
            return {}

        candidate_code = extract_python_code(response)
        script = "\n".join(
            [
                "import json",
                f"candidate = {json.dumps(candidate_code)}",
                f"tests = {json.dumps(tests)}",
                f"entrypoint = {json.dumps(entrypoint)}",
                "scope = {}",
                "try:",
                "    exec(candidate, scope, scope)",
                "    fn = scope.get(entrypoint)",
                "    if not callable(fn):",
                "        raise RuntimeError(f'Missing callable: {entrypoint}')",
                "    passed = 0",
                "    for expr in tests:",
                "        exec(expr, scope, scope)",
                "        passed += 1",
                "    print(json.dumps({'pass_rate': passed / len(tests) if tests else 0.0, 'passed': passed, 'total': len(tests)}))",
                "except Exception as exc:",
                "    print(json.dumps({'pass_rate': 0.0, 'passed': 0, 'total': len(tests), 'error': repr(exc)}))",
            ]
        ) + "\n"

        try:
            sandbox = modal.Sandbox.create(
                "python",
                "-c",
                script,
                app=sandbox_app(),
                image=sandbox_image,
                timeout=timeout_sec,
                cpu=1.0,
                memory=1024,
            )
            stdout = sandbox.stdout.read()
            stderr = sandbox.stderr.read()
            exit_code = sandbox.wait()
        except Exception as exc:
            candidate_bytes = len(candidate_code.encode("utf-8"))
            reference_bytes = len(reference_solution.encode("utf-8"))
            return 0.0, {
                "pass_rate": 0.0,
                "candidate_bytes": candidate_bytes,
                "reference_bytes": reference_bytes,
                "ratio": 0.0,
                "sandbox_exit_code": -1,
                "sandbox_stderr": repr(exc),
            }

        payload = last_json_line(stdout)
        pass_rate = float(payload.get("pass_rate", 0.0))
        candidate_bytes = len(candidate_code.encode("utf-8"))
        reference_bytes = len(reference_solution.encode("utf-8"))
        ratio = min(2.0, reference_bytes / max(candidate_bytes, 1))
        size_bonus = max(0.0, ratio - 1.0)
        reward = pass_rate * (1.0 + (length_bonus_weight * size_bonus))
        return reward, {
            "pass_rate": pass_rate,
            "candidate_bytes": candidate_bytes,
            "reference_bytes": reference_bytes,
            "ratio": ratio,
            "sandbox_exit_code": exit_code,
            "sandbox_stderr": stderr.strip(),
        }

    raw_label = getattr(sample, "label", None)
    if isinstance(raw_label, dict):
        label = raw_label
    elif raw_label:
        try:
            label = json.loads(raw_label)
        except json.JSONDecodeError:
            label = {}
    else:
        label = {}

    tests = label.get("tests")
    entrypoint = label.get("entrypoint")
    reference_solution = label.get("reference_solution")
    if (
        not isinstance(tests, list)
        or not tests
        or not isinstance(entrypoint, str)
        or not entrypoint
        or not isinstance(reference_solution, str)
        or not reference_solution
    ):
        sample_metadata = getattr(sample, "metadata", None)
        if not isinstance(sample_metadata, dict):
            sample_metadata = {}
        sample_metadata["code_golf"] = {"pass_rate": 0.0, "reason": "missing_label"}
        sample.metadata = sample_metadata
        return 0.0

    score_fn = globals().get("score_code_with_sandbox") or _inline_score_code_with_sandbox

    reward, metadata = await asyncio.to_thread(
        score_fn,
        sample.response,
        tests=tests,
        entrypoint=entrypoint,
        reference_solution=reference_solution,
    )
    sample_metadata = getattr(sample, "metadata", None)
    if not isinstance(sample_metadata, dict):
        sample_metadata = {}
    sample_metadata["code_golf"] = metadata
    sample.metadata = sample_metadata
    return float(reward)

## Serve and evaluate the base model

In [ ]:
base_model = Qwen3_4B()
base_deployment: ModelDeployment = DeploymentConfig(model=base_model).serve()
print(f"Base model URL: {base_deployment.url}")

def eval_fn(example: dict, response: str) -> EvalRowResult:
    score, metadata = score_code_with_sandbox(
        response,
        tests=example["tests"],
        entrypoint=example["entrypoint"],
        reference_solution=example["reference_solution"],
    )
    return EvalRowResult(score=score, response=response, metadata=metadata)

eval_config = EvalConfig(
    dataset=eval_dataset,
    eval_fn=eval_fn,
    generate_kwargs={"chat_template_kwargs": {"enable_thinking": False}},
)
print("Running base eval...")
base_eval = eval_config.evaluate(base_deployment, debug=True)
print(f"Base mean reward: {base_eval.mean:.4f}")

## Train with SLIME and sandbox reward

The custom RM function is the same sandbox scorer used in eval.
This keeps train/eval reward logic aligned.

In [ ]:
training_run = TrainConfig(
    model=Qwen3_4B(),
    dataset=train_dataset,
    recipe=SlimeRecipe(
        wandb=WandbConfig(project="gym-tutorial", group="qwen3-4b-code-golf"),
        custom_rm_function=code_golf_rm,
        num_rollout=10,
        rollout_batch_size=32,
        n_samples_per_prompt=8,
        global_batch_size=256,
        rollout_max_response_len=1024,
        eval_max_response_len=1024,
        n_samples_per_eval_prompt=8,
        rollout_temperature=0.9,
        max_tokens_per_gpu=4096,
        save_interval=10,
        apply_chat_template_kwargs='{"enable_thinking": false}',
        image_overlay=lambda image: image.run_commands(
            "uv pip install --system modal>=1.2.0 datasets>=3.0.0",
        ),
    ),
)
print("Starting training...")
train_result = training_run.train()
print(f"Training run id: {train_result.training_run_id}")

## Evaluate the trained checkpoint

In [ ]:
checkpoint = list_checkpoints(train_result.training_run_id)[-1]
trained_deployment = DeploymentConfig(
    model=Qwen3_4B(),
    checkpoint=checkpoint,
    app_name="qwen3-4b-code-golf-serve",
    served_model_name="qwen3-4b-code-golf",
).serve()
print(f"Trained model URL: {trained_deployment.url}")

trained_eval = eval_config.evaluate(trained_deployment, debug=True)
print(f"Trained mean reward: {trained_eval.mean:.4f}")
print(f"Base mean reward:    {base_eval.mean:.4f}")